In [ ]:
import os
import random
import numpy as np
import tensorflow as tf
import rasterio
import wandb


from wandb.integration.keras import WandbMetricsLogger, WandbModelCheckpoint
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
wandb.init(
    project="oil-spill-detection",
    name="CNN-6Conv-32Filters-512x512",
    config={
        "input_shape": (512, 512, 2),
        "conv_layers": 6,
        "filters": 32,
        "kernel_size": (3, 3),
        "pool_size": (2, 2),
        "dense_units": [20, 20],
        "activation": "relu",
        "optimizer": "Adam",
        "loss": "binary_crossentropy",
        "epochs": 50,
        "batch_size": 8
    }
)

In [ ]:
IMG_SIZE = (512, 512)
BATCH_SIZE = 8
EPOCHS = 5
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# BASE_PATH = "/Volumes/Windows8_OS/Dataset/Dataset-OG"
BASE_PATH = "/Volumes/Windows8_OS/Dataset/Dataset-OG"

TRAIN_IMG_DIR = os.path.join(BASE_PATH, "Train", "Images")
TEST_IMG_DIR  = os.path.join(BASE_PATH, "Test", "Images")  # if exists

In [ ]:
def load_sar_tiff(path):
    def _read(p):
        with rasterio.open(p.decode()) as src:
            vv = src.read(1).astype(np.float32)
            vh = src.read(2).astype(np.float32)

        vv = np.clip(vv, -35, 5)
        vh = np.clip(vh, -40, 0)

        vv = (vv + 35) / 40
        vh = (vh + 40) / 40

        img = np.stack([vv, vh], axis=-1)
        img = tf.image.resize(img, IMG_SIZE).numpy()
        return img

    img = tf.numpy_function(_read, [path], tf.float32)
    img.set_shape([512, 512, 2])
    return img

In [ ]:
def augment(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)

    k = tf.random.uniform([], 0, 4, dtype=tf.int32)
    image = tf.image.rot90(image, k)

    tx = tf.random.uniform([], -0.05, 0.05)
    ty = tf.random.uniform([], -0.05, 0.05)

    image = tf.roll(
        image,
        shift=[
            tf.cast(tx * IMG_SIZE[0], tf.int32),
            tf.cast(ty * IMG_SIZE[1], tf.int32)
        ],
        axis=[0, 1]
    )

    return image, label

In [ ]:
def build_balanced_dataset(images_root, target_per_class=1200):
    oil_dir = os.path.join(images_root, "Oil")
    no_oil_dir = os.path.join(images_root, "No_Oil")
    lookalike_dir = os.path.join(images_root, "Lookalike")

    oil_files = sorted([os.path.join(oil_dir, f) for f in os.listdir(oil_dir) if f.endswith(".tif")])
    no_oil_files = sorted([os.path.join(no_oil_dir, f) for f in os.listdir(no_oil_dir) if f.endswith(".tif")])
    lookalike_files = sorted([os.path.join(lookalike_dir, f) for f in os.listdir(lookalike_dir) if f.endswith(".tif")])

    combined_no_oil = no_oil_files + lookalike_files
    random.shuffle(combined_no_oil)

    oil_files = oil_files[:target_per_class]
    combined_no_oil = combined_no_oil[:target_per_class]

    paths = oil_files + combined_no_oil
    labels = [1]*len(oil_files) + [0]*len(combined_no_oil)

    return np.array(paths), np.array(labels)


In [ ]:
def build_test_dataset(images_root):
    oil_dir = os.path.join(images_root, "Oil")
    no_oil_dir = os.path.join(images_root, "No_Oil")
    lookalike_dir = os.path.join(images_root, "Lookalike")

    oil_files = [os.path.join(oil_dir, f) for f in os.listdir(oil_dir) if f.endswith(".tif")]
    no_oil_files = [os.path.join(no_oil_dir, f) for f in os.listdir(no_oil_dir) if f.endswith(".tif")]
    lookalike_files = [os.path.join(lookalike_dir, f) for f in os.listdir(lookalike_dir) if f.endswith(".tif")]

    paths = oil_files + no_oil_files + lookalike_files
    labels = (
        [1] * len(oil_files) +
        [0] * len(no_oil_files) +
        [0] * len(lookalike_files)
    )

    return np.array(paths), np.array(labels)

In [ ]:
from sklearn.model_selection import train_test_split

all_paths, all_labels = build_balanced_dataset(TRAIN_IMG_DIR)

print("Total balanced samples:", len(all_paths))

In [ ]:
train_paths, val_paths, train_labels, val_labels = train_test_split(
    all_paths,
    all_labels,
    train_size=0.66,
    stratify=all_labels,
    random_state=SEED
)

print("Train samples:", len(train_paths))
print("Val samples:", len(val_paths))

print("Train Oil:", np.sum(train_labels == 1))
print("Train No_Oil:", np.sum(train_labels == 0))
print("Val Oil:", np.sum(val_labels == 1))
print("Val No_Oil:", np.sum(val_labels == 0))

In [ ]:
test_paths, test_labels = build_test_dataset(TEST_IMG_DIR)

print("Test samples:", len(test_paths))  # 450
print("Test Oil:", np.sum(test_labels == 1))        # 150
print("Test No_Oil:", np.sum(test_labels == 0))     # 300

In [ ]:
def make_dataset(paths, labels, augment_fn=None, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    if shuffle:
        ds = ds.shuffle(
            buffer_size=len(paths),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    ds = ds.map(
        lambda x, y: (load_sar_tiff(x), y),
        num_parallel_calls=1  # rasterio-safe
    )

    if augment_fn is not None:
        ds = ds.map(augment_fn, num_parallel_calls=1)

    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds


In [ ]:
train_ds = make_dataset(
    train_paths,
    train_labels,
    augment_fn=augment,
    shuffle=True
)

val_ds = make_dataset(
    val_paths,
    val_labels,
    augment_fn=None,
    shuffle=False
)

test_ds = make_dataset(
    test_paths,
    test_labels,
    augment_fn=None,
    shuffle=False
)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

def build_cnn():
    model = Sequential()

    for i in range(6):
        if i == 0:
            model.add(Conv2D(
                filters=32,
                kernel_size=(3, 3),
                activation="relu",
                padding="same",
                input_shape=(512, 512, 2)
            ))
        else:
            model.add(Conv2D(
                filters=32,
                kernel_size=(3, 3),
                activation="relu",
                padding="same"
            ))

        model.add(MaxPooling2D(pool_size=(2, 2)))


    model.add(Flatten())

    model.add(Dense(20, activation="relu"))
    model.add(Dense(20, activation="relu"))

    model.add(Dense(1, activation="sigmoid"))

    model.compile(
        optimizer=tf.keras.optimizers.Adam(),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

# Hold On

In [ ]:
MODEL_PATH   = "CNN/Models/latest.keras"
WEIGHTS_PATH = "CNN/Weights/weights_epoch_{epoch:03d}.weights.h5"

os.makedirs("CNN", exist_ok=True)

if os.path.exists(MODEL_PATH):
    print("🔁 Resuming training from saved model...")
    model = tf.keras.models.load_model(MODEL_PATH)
else:
    print("🆕 Starting training from scratch...")
    model = build_cnn()

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True
)

# Save FULL MODEL (for resume)
model_ckpt = tf.keras.callbacks.ModelCheckpoint(
    filepath=MODEL_PATH,
    save_best_only=False,
    verbose=1
)

# Save WEIGHTS separately (for analysis)
weights_ckpt = tf.keras.callbacks.ModelCheckpoint(
    filepath=WEIGHTS_PATH,
    save_weights_only=True,
    save_best_only=False,
    verbose=0
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[
        early_stop,
        model_ckpt,
        weights_ckpt,
        WandbMetricsLogger(log_freq="epoch")
    ]
)


# Continue

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True
)

checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    filepath="CNN_checkpoints/model_epoch_{epoch}.weights.h5",
    save_weights_only=True,
    save_best_only=False
)

In [ ]:
model = build_cnn()
model.load_weights("CNN_checkpoints/model_epoch_10.weights.h5")

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    initial_epoch=10,
    callbacks=[
        early_stop, 
        checkpoint_cb, 
        WandbMetricsLogger(log_freq="epoch")
    ]
)

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

y_true = []
y_pred = []

for x_batch, y_batch in test_ds:
    preds = model.predict(x_batch, verbose=0)
    preds = (preds > 0.35).astype(int)   # threshold = 0.5

    y_true.extend(y_batch.numpy())
    y_pred.extend(preds.flatten())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

In [ ]:
cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:")
print(cm)

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

In [ ]:
val_loss, val_acc = model.evaluate(val_ds)
print(f"Val Loss: {val_loss:.4f}")
print(f"Val Accuracy: {val_acc:.4f}")